In [135]:
import sys
import models
from omegaconf import OmegaConf
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
import scipy
import torch
import sklearn as sk
import mne

In [3]:
import os
os.chdir('../..')
os.getcwd()

'/Data/EEG-Visual-Experiment'

In [4]:
from Scripts.Data_Loader import EIRDataset

In [10]:
eir_dataset = EIRDataset('Generated/Data_Train', 'geometric')

Loading .fif files: 100%|██████████| 840/840 [00:40<00:00, 20.83it/s]


In [62]:
sum(idx)

np.int64(40)

In [75]:
np.array([idx]*63).shape

(63, 513)

In [69]:
f_filtered = all_freqs[idx]
f_filtered

array([ 0.9765625,  1.953125 ,  2.9296875,  3.90625  ,  4.8828125,
        5.859375 ,  6.8359375,  7.8125   ,  8.7890625,  9.765625 ,
       10.7421875, 11.71875  , 12.6953125, 13.671875 , 14.6484375,
       15.625    , 16.6015625, 17.578125 , 18.5546875, 19.53125  ,
       20.5078125, 21.484375 , 22.4609375, 23.4375   , 24.4140625,
       25.390625 , 26.3671875, 27.34375  , 28.3203125, 29.296875 ,
       30.2734375, 31.25     , 32.2265625, 33.203125 , 34.1796875,
       35.15625  , 36.1328125, 37.109375 , 38.0859375, 39.0625   ])

In [87]:
from tqdm import tqdm
powers = []
labels = []
for i in tqdm(eir_dataset):
    labels.append(i[3])
    power = SFT.stft(i[0].get_data())
    powers.append(power[:,idx, :])

100%|██████████| 840/840 [05:16<00:00,  2.66it/s]


In [92]:
from tqdm import tqdm
powers2 = []
labels2 = []
subjects = []
trials = []
for i in tqdm(eir_dataset):
    subjects.append(i[2]['subject_id'])
    trials.append(i[2]['trial_id'])

100%|██████████| 840/840 [00:00<00:00, 6419.88it/s]


In [119]:
from Scripts import Selectors_From_Dataset as sel
#X, img, y = sel.get_sample_choosen_trial(eir_dataset, subj_id = [12, 12], choosen_trial = [1, 2])
X, img, y = sel.get_sample(eir_dataset)

In [121]:
from scipy.signal import ShortTimeFFT, windows

# Желаемый частотный диапазон и шаг
freq_start = 0.5  # Гц
freq_end = 40   # Гц
freq_step = 1   # Гц

# Расчет необходимых параметров
fs = 1000  # Частота дискретизации
desired_freqs = np.arange(freq_start, freq_end + freq_step, freq_step)

# nfft должен быть степенью двойки для эффективности
# Δf = fs / nfft -> nfft = fs / freq_step
nfft_needed = int(fs / freq_step)  # 1000/1 = 1000
# Ближайшая степень двойки
nfft = 2 ** int(np.ceil(np.log2(nfft_needed)))  # 1024

nperseg = 64
hop = 50
noverlap = nperseg - hop  # 64 - 10 = 54

window = windows.hann(nperseg, sym=False)
SFT = ShortTimeFFT(window, hop=hop, fs=fs, mfft=nfft, 
                   scale_to='magnitude')

all_freqs = SFT.f

# Выбираем только нужный диапазон
idx = (all_freqs >= freq_start) & (all_freqs <= freq_end)
#idx = np.array([idx]*63)

In [122]:
from tqdm import tqdm
powers = []
for i in tqdm(X):
    power = SFT.stft(i)
    powers.append(power[:,idx, :])

100%|██████████| 840/840 [05:24<00:00,  2.58it/s]


In [124]:
X.shape

(840, 63, 16001)

In [125]:
np.array(powers).shape

(840, 63, 40, 321)

In [126]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [129]:
## loading model
device = 'cuda'
def build_model(cfg):
    ckpt_path = cfg.upstream_ckpt
    init_state = torch.load(ckpt_path)
    upstream_cfg = init_state["model_cfg"]
    print(upstream_cfg)
    upstream = models.build_model(upstream_cfg)
    return upstream

def load_model_weights(model, states, multi_gpu):
    if multi_gpu:
        model.module.load_weights(states)
    else:
        model.load_weights(states)

ckpt_path = 'EEG-Visual-Experiment/Supplementary/stft_large_pretrained.pth'
cfg = OmegaConf.create({"upstream_ckpt": ckpt_path})
print(cfg)
tf_model = build_model(cfg)
tf_model.to(device)
init_state = torch.load(ckpt_path)
load_model_weights(tf_model, init_state['model'], False)


print(tf_model)

class MaskedTFClassifier(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, num_classes):
        super(MaskedTFClassifier, self).__init__()
        self.base_model = base_model

        
        self.base_model.spec_prediction_head = torch.nn.Identity()

       
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 768),
            torch.nn.LayerNorm(768),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),

            torch.nn.Linear(768, 512),
            torch.nn.LayerNorm(512),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),

            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),

            torch.nn.Linear(256, 64),
            torch.nn.ReLU(),

            torch.nn.Linear(64, num_classes)  # No softmax here (handled in loss)
        )

    def forward(self, input_specs, src_key_mask,inter_state):

        transformer_output = self.base_model(
            input_specs=input_specs,
            src_key_mask=src_key_mask,
            intermediate_rep=True
        )  # shape: [batch, seq, 768]
        
        if inter_state:
            return transformer_output

        pooled = transformer_output.mean(dim=1)  # [batch, 768]

        return self.classifier(pooled)
    
# selected_model = MaskedTFClassifier(tf_model, num_classes=3)


def training(model,criteria,optimizer,training_data,num_epochs):
    train_losses = []
    for epoch in range(num_epochs):    
        model.train()
        train_loss = 0.0
        
        for param in model.base_model.spec_prediction_head.parameters():
            param.requires_grad = False
            
        # if epoch >= 200:
        #     for param in model.base_model.parameters():
        #         param.requires_grad = False
                
        for batch_X,batch_mask, batch_y in training_data:
            batch_X, batch_mask, batch_y = batch_X.to(device),batch_mask.to(device),  batch_y.to(device).long() 
            batch_mask = torch.zeros((batch_X.shape[:2])).bool().to(device)
            
            outputs = model(batch_X, batch_mask,inter_state=False)
            # print(outputs.shape)
            loss = criteria(outputs,batch_y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss +=loss.item()

            
        train_loss /=len(training_data)
        train_losses.append(train_loss)

        if (epoch+1)% 5 ==0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss = {train_loss:.4f}')
            
    plt.figure(figsize=(10,6))
    plt.plot(train_losses,label = "Training Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss over epochs")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return model
            
            
            
        
    

{'upstream_ckpt': 'EEG-Visual-Experiment/Supplementary/stft_large_pretrained.pth'}
{'name': 'masked_tf_model', 'hidden_dim': 768, 'layer_dim_feedforward': 3072, 'layer_activation': 'gelu', 'nhead': 12, 'encoder_num_layers': 6, 'input_dim': 40}
MaskedTFModel(
  (input_encoding): TransformerEncoderInput(
    (in_proj): Linear(in_features=40, out_features=768, bias=True)
    (positional_encoding): PositionalEncoding()
    (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1)

In [131]:
from sklearn.model_selection import train_test_split
X = np.array(powers)
X_train, X_val, y_train, y_val = train_test_split(X, labels, test_size=0.25, 
    random_state=42,
    stratify=labels)

In [138]:
X_train.shape

(630, 63, 40, 321)

In [ ]:
X_Train.reshape(X_Train.shape[0]*X_Train.shape[1], X_Train.shape[2], X_Train.shape[3])

In [137]:
# Почему маска фиксированная и нет валидации??

# Transposing to shuffle timestepms and freq bins
X_trainT, X_valT = X_train.transpose(0, 2, 1), X_val.transpose(0, 2, 1)
print(X_trainT.shape, X_valT.shape)

# To tensor
X_trainT, X_valT = torch.FloatTensor(X_trainT), torch.FloatTensor(X_valT)
X_trainT.to(device)
X_valT.to(device)
print(X_trainT.shape, X_valT.shape)


# Loading y
y_train, y_val = torch.Tensor(y_train.reshape(-1)), torch.Tensor(y_val.reshape(-1))
print(y_train.shape, y_val.shape)
# creating mask
mask = torch.rand(X.shape[0], X.shape[1]) < 0.15
print(mask.shape)
# print(y)

ValueError: axes don't match array

In [ ]:
train_dataset = torch.utils.data.TensorDataset(X,mask,y)
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=32,shuffle=True)

selected_model = MaskedTFClassifier(tf_model, num_classes=len(np.unique(y)))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
selected_model.to(device)

selected_criteria  = torch.nn.CrossEntropyLoss()
selected_optimizer = torch.optim.AdamW(selected_model.parameters(),lr=1e-4,weight_decay=1e-2)

trained_model = training(model= selected_model,
                                 criteria = selected_criteria,
                                 optimizer=selected_optimizer,
                                 training_data= train_loader,
                                 # validation_data = val_loader,
                                 num_epochs = 270)